In [ ]:
import re
from collections import defaultdict
import plotly.graph_objects as go
import numpy as np
import wandb

In [ ]:
MAP = {
    'random': 'Random',
    'kmeans': 'K-Means',
    'dpp': 'DPP',
    'kcenter': 'K-Center',
    'gist': 'GIST',
    'mtt': 'MTT'
}
DATASETS = [
    'RaspiCar',
    'ActionSense',
    'MMCows'
]

# Create Plots and Tables

In [ ]:
%%time

accuracies = defaultdict(list)

for r in wandb.Api().runs(wandb.Api().default_entity +  '/DatasetDistillation_Training'):
    # only query if it has the right prefix / is one of the used datasets
    if not [d for d in DATASETS if r.name.startswith(d)]:
        continue
    
    hist = list(r.scan_history())
    test_acc = hist[-1]['test_acc']
    
    accuracies[r.name].append(test_acc)

full_accuracies_mean = {}
full_accuracies_std = {}

for k in accuracies:
    full_accuracies_mean[k] = np.array(accuracies[k]).mean().item() * 100
    full_accuracies_std[k] = np.array(accuracies[k]).std().item() * 100

In [ ]:
mtt_results = []

for r in wandb.Api().runs(wandb.Api().default_entity + '/DatasetDistillation'):    
    hist = list(r.scan_history())

    try:
        dataset, spc, modality = r.name.split('_')
        spc = int(spc.strip('SPC'))
    except:
        print(f'Not able to to parse {r.name} ({r.id})')
        continue

    acc = hist[-1]['Max_Accuracy'] * 100
    std = hist[-1]['Max_Std'] * 100

    mtt_results.append({
        'dataset': dataset,
        'spc': spc,
        'methods': {'MTT': {'mean': acc, 'std': std}},
        'modality': modality
    })

In [ ]:
def merge_results(orig_results, mtt_results):
    for idx in range(len(results)):
        element = results[idx]

        for mtt in mtt_results:
            dataset = mtt['dataset']
            spc = mtt['spc']
            modality = mtt['modality']
        
            if element['dataset'] == dataset and element['spc'] == spc and element['modality'] == modality:
                
                element['methods']['MTT'] = mtt['methods']['MTT']
                results[idx] = element
    
    return results

In [ ]:
def parse_logs(log_fn):
    with open(log_fn, "r", encoding="utf-8") as f:
        text = f.read()
    
    block_pattern = re.compile(
        r"Summary of all methods \(Test Accuracy\) for\s+"
        r"(?P<dataset>.+?)\s+SPC=(?P<spc>\d+):\s*"
        r"(?P<body>.*?)(?=\n\s*\n|#{10,}|\Z)",
        re.DOTALL
    )
    
    line_pattern = re.compile(
        r"^(?P<method>random|kmeans|DPP|kcenter|gist):\s+"
        r"mean=(?P<mean>\d+(?:\.\d+)?)\s*\|\s*"
        r"std=(?P<std>\d+(?:\.\d+)?)",
        re.MULTILINE
    )
    
    all_results = []
    
    for block in block_pattern.finditer(text):
        block_result = {
            "dataset": block.group("dataset"),
            "spc": int(block.group("spc")),
            "methods": {}
        }
        if ' ' in block_result['dataset']:
            full_name = block_result['dataset']
            dataset, modality = block_result['dataset'].split(' ')
            modality = modality.capitalize()
            block_result['modality'] = modality
            block_result['dataset'] = dataset
        else:
            block_result['modality'] = 'MM'
    
        body = block.group("body")
    
        for line in line_pattern.finditer(body):
            method = line.group("method").lower()
            block_result["methods"][MAP[method]] = {
                "mean": float(line.group("mean")) * 100,
                "std": float(line.group("std")) * 100,
            }
    
        all_results.append(block_result)

    return all_results

In [ ]:
def create_plot(data, modality, dataset):
    fig = go.Figure()
    
    colors  = {1: 'blue', 10: 'red', 50: 'green'}
    symbols = {1: 'circle', 10: 'square', 50: 'triangle-up'}
    styles  = {1: 'solid', 10: 'dash', 50: 'dashdot'}
    
    for d in data:
        if d['modality'] != modality:
            continue
    
        spc = d['spc']
        
        x     = list(d['methods'].keys())
        y     = [d['methods'][x]['mean'] for x in x]
        y_std = [d['methods'][x]['std'] for x in x]
    
        text_labels = [f"{val:.2f}" for val in y]
    
        fig.add_trace(
            go.Scatter(
                x=x,
                y=y,
                mode='markers+text',
                line_dash=styles[spc],
                name=f'SPC={d["spc"]}',
                text=text_labels,
                textposition='top center',
                textfont=dict(color=colors[spc], size=24),
                marker=dict(symbol=symbols[spc], size=14)
            )
        )
    
    full_acc = full_accuracies_mean[d['dataset']  + '_' + modality]
    fig.add_hline(
        y=full_acc,
        line_dash="dot",
        line_color="purple",
        annotation_text=f'Full: {full_acc:.2f}',
        annotation_position="top right",
        annotation_font_color="purple",
        annotation_font_size=24
    )
    
    fig.update_layout(
        width=1440//2,
        height=1280//2,
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(
            size=28,
            color="black"
        ),
        yaxis=dict(
            title=dict(text="Max. Accuracy [%]") if dataset == 'raspicar' else ''
        ),
        legend=dict(
            orientation="h",
            x=0.5,
#            y=-0.1,
            y=1.12,
            xanchor="center",
            yanchor="top",
            bgcolor="white",
        ),
        margin=dict(l=0, r=0, b=0, t=0)
    )
    
    fig.update_xaxes(
        showline=True,
        linecolor='black',
        linewidth=2,
        mirror=True,
        ticks='outside',
        tickcolor='black',
        showgrid=True,
        gridcolor='lightgrey',
        griddash='dot',
        zeroline=False
    )
    
    fig.update_yaxes(
        showline=True,
        linecolor='black',
        linewidth=2,
        mirror=True,
        ticks='outside',
        tickcolor='black',
        showgrid=True,
        gridcolor='lightgrey',
        griddash='dot',
        zeroline=False
    )

    return fig

In [ ]:
def print_table(results):
    table = {1: {}, 10: {}, 50:{}, 'Full': {}}
    
    for r in results:
        dataset = r['dataset']
        spc = r['spc']
        modality = r['modality']
        key = dataset + '_' + modality
        
        table[spc][modality] = r['methods']['MTT']
        table['Full'][modality] = {
            'mean': full_accuracies_mean[key],
            'std': full_accuracies_std[key]
        }
    
    print(f"      {'Image':<14}  {'Sensor':<14}  {'Multimodal':<14}")
    for spc, line in table.items():
        print(
            f'{spc}'.ljust(6) +
            f'{line["Image"]["mean"]:.2f} ± {line["Image"]["std"]:.2f}'.ljust(14) + '| ' + 
            f'{line["Sensor"]["mean"]:.2f} ± {line["Sensor"]["std"]:.2f}'.ljust(14) + '| ' +
            f'{line["MM"]["mean"]:.2f} ± {line["MM"]["std"]:.2f}'.ljust(14)
        )

In [ ]:
modality = 'MM'

for ds in DATASETS:
    ds = ds.lower()
    fn = f'../task_logs/baselines-mmsconvb-{ds}.out'
    
    results = parse_logs(fn)
    results = merge_results(results, mtt_results)

    print(f'\n{ds} - {modality}\n\n')
    print_table(results)
    fig = create_plot(results, modality, ds)
    fig.show()
    fig.write_image(f'plots/baselines-{ds}-{modality}.pdf')

# Show Sample MMCows Image

In [ ]:
from utils import load_mmcows_data, MMCowsDataset
from types import SimpleNamespace
import plotly.express as px

In [ ]:
args = SimpleNamespace(image_size=(224, 224), unimodal='image')

In [ ]:
df = load_mmcows_data(args)

num_classes = df['behavior'].nunique()
ds = MMCowsDataset(df, args.sens_cols, num_classes, args.image_size, args.unimodal)

In [ ]:
fig = px.imshow(ds[0][0][0].numpy().transpose(1,2, 0))
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.update_layout(width=500, height=500, margin=dict(l=0, r=0, b=0, t=0))
fig.write_image('plots/mmcows-sample.png')